# Stage 6 & 7: Algorithm 1 — Logistic Regression

**Owner:** Member A  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/logistic_regression_tuned.pkl`  

## Scope
Logistic Regression is selected as the **linear baseline model** for late delivery risk prediction.
It models the log-odds of an order being late as a linear combination of input features.
Key advantages: interpretable coefficients, fast training, handles class imbalance via `class_weight='balanced'`.

This notebook:
1. Loads the final modeling datasets from `data/final/`
2. Establishes a Dummy Classifier lower-bound baseline
3. Trains a default Logistic Regression baseline
4. Performs 5-Fold Stratified K-Fold hyperparameter tuning on `X_train` only
5. Evaluates the tuned model on `X_val` (never on `X_test`)
6. Reports feature coefficients and exports the tuned model artifact


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)

FINAL_DIR  = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train : {X_train.shape} | Late rate: {y_train.mean()*100:.2f}%')
print(f'X_val   : {X_val.shape}   | Late rate: {y_val.mean()*100:.2f}%')


X_train : (67529, 67) | Late rate: 9.03%
X_val   : (14470, 67)   | Late rate: 5.34%


## 2. Dummy Classifier Baseline
A Dummy Classifier that always predicts the majority class (on-time) sets the absolute lower bound.
Any real model must beat this to be useful.

In [2]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_probs = dummy.predict_proba(X_val)[:, 1]

print('=== DUMMY CLASSIFIER BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, dummy_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, dummy_probs):.4f}')
print('A PR-AUC equal to the positive class rate (~0.053) is the baseline to beat.')


=== DUMMY CLASSIFIER BASELINE ===
Val ROC-AUC : 0.5000
Val PR-AUC  : 0.0534
A PR-AUC equal to the positive class rate (~0.053) is the baseline to beat.


## 3. Default Logistic Regression Baseline
Before tuning, we establish an untuned Logistic Regression baseline to measure the benefit of hyperparameter search.

In [3]:
lr_default = LogisticRegression(max_iter=1000, random_state=42)
lr_default.fit(X_train, y_train)
default_probs = lr_default.predict_proba(X_val)[:, 1]

print('=== DEFAULT LOGISTIC REGRESSION BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, default_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, default_probs):.4f}')


=== DEFAULT LOGISTIC REGRESSION BASELINE ===
Val ROC-AUC : 0.7707
Val PR-AUC  : 0.1600


## 4. Hyperparameter Tuning — 5-Fold Stratified GridSearchCV
**Parameters tuned:**
- `C`: Inverse regularization strength. Smaller = stronger regularization (prevents overfitting).
- `class_weight`: `balanced` adjusts class weights inversely to class frequencies to handle the 11:1 imbalance.

**Strategy:** 5-Fold Stratified K-Fold cross-validation on `X_train` only. Validation set `X_val` is NEVER seen during tuning.
**Scoring metric:** `average_precision` (PR-AUC) — optimal for imbalanced classification tasks.

In [4]:
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'class_weight': ['balanced', None]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42),
    param_grid, scoring='average_precision', cv=cv, n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train)

print(f'Best Parameters     : {grid.best_params_}')
print(f'Best CV PR-AUC      : {grid.best_score_:.4f}')
best_lr = grid.best_estimator_


Fitting 5 folds for each of 8 candidates, totalling 40 fits


Best Parameters     : {'C': 0.1, 'class_weight': None}
Best CV PR-AUC      : 0.2204


## 5. Validation Set Evaluation

In [5]:
val_probs = best_lr.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print('=== TUNED LOGISTIC REGRESSION — VALIDATION SET METRICS ===')
print(f'Val ROC-AUC        : {roc_auc_score(y_val, val_probs):.4f}')
print(f'Val PR-AUC         : {average_precision_score(y_val, val_probs):.4f}')
print(f'Val F1-Score (0.5) : {f1_score(y_val, val_preds):.4f}')
print(f'Val Recall   (0.5) : {recall_score(y_val, val_preds):.4f}')
print(f'Val Precision(0.5) : {precision_score(y_val, val_preds):.4f}')

print('\nClassification Report:')
print(classification_report(y_val, val_preds, target_names=['On-Time (0)', 'Late (1)']))

cm = confusion_matrix(y_val, val_preds)
print(f'Confusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')


=== TUNED LOGISTIC REGRESSION — VALIDATION SET METRICS ===
Val ROC-AUC        : 0.7718
Val PR-AUC         : 0.1606
Val F1-Score (0.5) : 0.0026
Val Recall   (0.5) : 0.0013
Val Precision(0.5) : 0.1667

Classification Report:
              precision    recall  f1-score   support

 On-Time (0)       0.95      1.00      0.97     13697
    Late (1)       0.17      0.00      0.00       773

    accuracy                           0.95     14470
   macro avg       0.56      0.50      0.49     14470
weighted avg       0.90      0.95      0.92     14470

Confusion Matrix:
  TN=13,692  FP=5
  FN=772  TP=1


## 6. Feature Coefficients (Model Interpretability)
Logistic Regression coefficients reveal which features most strongly drive the late delivery probability.
Positive coefficient = feature increases probability of late delivery.
Negative coefficient = feature decreases probability of late delivery.

In [6]:
coef_series = pd.Series(best_lr.coef_[0], index=X_train.columns)
top_positive = coef_series.sort_values(ascending=False).head(10)
top_negative = coef_series.sort_values(ascending=True).head(5)

print('Top 10 Features INCREASING Late Delivery Risk (positive coefficients):')
for feat, coef in top_positive.items():
    print(f'  {feat:<45} : {coef:+.4f}')

print('\nTop 5 Features DECREASING Late Delivery Risk (negative coefficients):')
for feat, coef in top_negative.items():
    print(f'  {feat:<45} : {coef:+.4f}')


Top 10 Features INCREASING Late Delivery Risk (positive coefficients):
  purchase_month                                : +0.9786
  customer_state_target_enc                     : +0.4227
  customer_region__North                        : +0.4075
  is_cross_state                                : +0.2038
  customer_geolocation_missing                  : +0.1415
  missing_customer_coordinates                  : +0.1415
  approval_lag_hours                            : +0.1386
  minimum_distance_km                           : +0.1151
  installment_ratio                             : +0.1146
  average_freight_value                         : +0.1133

Top 5 Features DECREASING Late Delivery Risk (negative coefficients):
  purchase_quarter                              : -0.9624
  estimated_delivery_days                       : -0.7139
  customer_region__Southeast                    : -0.3038
  customer_region__South                        : -0.3000
  any_seller_same_state                       

## 7. Save Tuned Model Artifact

In [7]:
joblib.dump(best_lr, MODELS_DIR / 'logistic_regression_tuned.pkl')
print('Saved: models/logistic_regression_tuned.pkl')
print(f'Model type : {type(best_lr).__name__}')
print(f'Best C     : {best_lr.C}')
print(f'class_weight: {best_lr.class_weight}')


Saved: models/logistic_regression_tuned.pkl
Model type : LogisticRegression
Best C     : 0.1
class_weight: None
